In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
conn = sqlite3.connect('adventure_works_analysis.db')

In [3]:
customers = pd.read_csv('AdventureWorks_Customers.csv', encoding='latin-1')
sales_2015 = pd.read_csv('AdventureWorks_Sales_2015.csv')
sales_2016 = pd.read_csv('AdventureWorks_Sales_2016.csv')
sales_2017 = pd.read_csv('AdventureWorks_Sales_2017.csv')
products = pd.read_csv('AdventureWorks_Products.csv')
product_categories = pd.read_csv('AdventureWorks_Product_Categories.csv')
product_subcategories = pd.read_csv('AdventureWorks_Product_Subcategories.csv')
territories = pd.read_csv('AdventureWorks_Territories.csv')
returns = pd.read_csv('AdventureWorks_Returns.csv')

In [4]:
customers.to_sql('customers', conn, if_exists='replace')
sales = pd.concat([sales_2015, sales_2016, sales_2017], ignore_index=True)
sales.to_sql('sales', conn, if_exists='replace')
products.to_sql('products', conn, if_exists='replace')
product_categories.to_sql('product_categories', conn, if_exists='replace')
product_subcategories.to_sql('product_subcategories', conn, if_exists='replace')
territories.to_sql('territories', conn, if_exists='replace')
returns.to_sql('returns', conn, if_exists='replace')

1809

In [5]:
first_query = """ SELECT "CategoryName", ROUND(SUM("ProductPrice" * "OrderQuantity"), 2) as 'Revenue' FROM sales
JOIN products ON sales."ProductKey" = products."ProductKey"
JOIN product_subcategories ON products."ProductSubcategoryKey" = product_subcategories."ProductSubcategoryKey"
JOIN product_categories ON product_subcategories."ProductCategoryKey" = product_categories."ProductCategoryKey"
JOIN territories ON sales."TerritoryKey" = territories."SalesTerritoryKey"
GROUP BY "CategoryName" ORDER BY "Revenue" DESC """
revenue_by_categories = pd.read_sql(first_query, conn)

Bikes is the dominant revenue category at Adventure Works, generating over $23.6M — almost 95% of total revenue across 2015–2017. Accessories and Clothing contribute marginally at $0.9M and $0.4M respectively.

In [6]:
second_query = """ SELECT "Country", ROUND(SUM("ProductPrice" * "OrderQuantity"), 2) as 'Revenue' FROM sales
JOIN products ON sales."ProductKey" = products."ProductKey"
JOIN territories ON sales."TerritoryKey" = territories."SalesTerritoryKey" 
GROUP BY "Country" ORDER BY "Revenue" DESC """
revenue_by_country = pd.read_sql(second_query, conn)

The United States and Australia are the top two markets, together generating over $15.5M — approximately 62% of total revenue. European markets (United Kingdom, Germany, France) collectively contribute around $7.5M, with each country showing comparable performance between $2M and $3M.

In [7]:
third_query = """ SELECT "ProductName", "CategoryName", ROUND(SUM("ProductPrice" * "OrderQuantity"), 2) as 'Revenue' FROM sales
JOIN products ON sales."ProductKey" = products."ProductKey"
JOIN product_subcategories ON products."ProductSubcategoryKey" = product_subcategories."ProductSubcategoryKey"
JOIN product_categories ON product_subcategories."ProductCategoryKey" = product_categories."ProductCategoryKey"
GROUP BY "ProductName", "CategoryName" ORDER BY "Revenue" DESC LIMIT 10 """
revenue_by_product = pd.read_sql(third_query, conn)

In [8]:
fourth_query = """ SELECT a."CategoryName", ROUND(b."ReturnValue" / a."Revenue" * 100, 2) as "Return Rate"
FROM (SELECT "CategoryName", ROUND(SUM("ProductPrice" * "OrderQuantity"), 2) as 'Revenue' FROM sales
JOIN products ON sales."ProductKey" = products."ProductKey"
JOIN product_subcategories ON products."ProductSubcategoryKey" = product_subcategories."ProductSubcategoryKey"
JOIN product_categories ON product_subcategories."ProductCategoryKey" = product_categories."ProductCategoryKey"
GROUP BY "CategoryName") as a 
JOIN (SELECT "CategoryName", ROUND(SUM("ProductPrice" * "ReturnQuantity"), 2) as "ReturnValue" FROM returns
JOIN products ON returns."ProductKey" = products."ProductKey"
JOIN product_subcategories ON products."ProductSubcategoryKey" = product_subcategories."ProductSubcategoryKey"
JOIN product_categories ON product_subcategories."ProductCategoryKey" = product_categories."ProductCategoryKey"
GROUP BY "CategoryName") as b ON a."CategoryName" = b."CategoryName"
GROUP BY a."CategoryName" ORDER BY "Return Rate" DESC"""

return_rate = pd.read_sql(fourth_query, conn)

The return rate is consistent across all categories, ranging from 2.34% (Accessories) to 3.10% (Bikes). This suggests no systematic quality issues within any specific product category.

In [9]:
fifth_query = """ SELECT "FirstName", "LastName", ROUND(SUM("ProductPrice" * "OrderQuantity"), 2) as "Revenue" FROM sales
JOIN products ON sales."ProductKey" = products."ProductKey" 
JOIN customers ON sales."CustomerKey" = customers."CustomerKey" 
GROUP BY sales."CustomerKey", "FirstName", "LastName" ORDER BY "Revenue" DESC LIMIT 10 """
top_10_customers = pd.read_sql(fifth_query, conn)

The top customer, Maurice Shan, generated $12,408 in total revenue over 2015–2017. All top 10 customers fall within a narrow range of $10K–$12.5K, suggesting no single customer has a dominant share — revenue is distributed relatively evenly among the best buyers.